In [1]:
import gradio as gr
import pandas as pd

# 实时天气模块
from Weather.get_weather import *

In [2]:
# 自定义 CSS（让侧边栏更窄，整体更紧凑）
custom_css = """
<style>
    .sidebar {
        background-color: #f8f9fa;
        padding: 15px;
        border-right: 1px solid #e0e0e0;
    }
    .main-tabs {
        padding: 10px;
    }
    .gray-placeholder {
        color: #999;
        font-style: italic;
    }
    .warning-banner {
        background-color: #ffebee;
        border-left: 4px solid #f44336;
        padding: 10px;
        margin: 10px 0;
    }
</style>
"""

with gr.Blocks(title="光储充一体化智能预测平台", css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.HTML("<h1 style='text-align: center;'>☀️ 光储充一体化智能预测平台 🔋</h1>")
    gr.Markdown("*基于 TCN 的光伏出力预测 + 电动汽车充电负荷预测*")

    with gr.Row():
        # ========== 左侧窄边栏（控制面板） ==========
        with gr.Column(scale=1, elem_classes="sidebar"):
            gr.Markdown("### 🎮 控制面板")
            
            # 模型选择（包含占位符说明）
            model_choice = gr.Dropdown(
                choices=[
                    "光伏预测模型 (TCN) - 已完成",
                    "充电负荷模型 (占位符 - 返回昨日曲线)",
                ],
                value="光伏预测模型 (TCN) - 已完成",
                label="📊 当前预测模型",
                info="未完成的模型会返回模拟数据，接口已预留"
            )
            gr.Markdown("<p class='gray-placeholder'>💡 提示：充电负荷模型为占位版本，待集成正式模型</p>")
            
            # 手动输入区域
            gr.Markdown("### ⚡ 实时数据输入")
            manual_load = gr.Number(
                label="当前负荷 load_kw（手动模拟）",
                value=150.0,
                step=10.0
            )
            manual_pv = gr.Number(
                label="当前光伏出力（可选覆盖）",
                value=80.0,
                step=5.0
            )
            
            # 控制按钮
            refresh_btn = gr.Button("🔄 刷新预测与图表", variant="primary")
            gr.Markdown("---")
            gr.Markdown("**模型状态**")
            gr.Markdown("- ✅ 光伏预测模型: 已加载")
            gr.Markdown("- ⏳ 充电负荷模型: 占位符")

        # ========== 右侧主区域（多 Tab） ==========
        with gr.Column(scale=4, elem_classes="main-tabs"):
            with gr.Tabs():
                # ---------- Tab 1: 预测核心 ----------
                with gr.TabItem("📈 预测核心"):
                    # 曲线对比图（占位）
                    gr.Markdown("### 📊 负荷预测曲线对比")
                    gr.Markdown("*实测曲线（实线）vs 预测曲线（虚线）vs 置信区间（阴影）*")
                    gr.Plot(value=None, label="预测曲线图占位", show_label=False)
                    gr.Markdown("*图表示例：横轴时间（15min粒度），纵轴功率 (kW)*")
                    
                    # 特征敏感度 + 误差统计两列
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### 🔥 特征敏感度分析")
                            gr.Markdown("*当前预测受各特征影响程度*")
                            gr.Plot(value=None, label="热力图占位")
                        with gr.Column():
                            gr.Markdown("### 📉 误差统计")
                            gr.Markdown("*MAPE / RMSE 分段展示*")
                            # 修复：使用 pandas DataFrame 传入，不需要 rows 参数
                            error_data = pd.DataFrame([
                                ["00:00-06:00", "5.2", "12.3"],
                                ["06:00-12:00", "3.8", "8.9"],
                                ["12:00-18:00", "4.5", "10.1"],
                                ["18:00-24:00", "6.1", "15.4"]
                            ], columns=["时段", "MAPE (%)", "RMSE (kW)"])
                            gr.Dataframe(value=error_data, label="今日分段误差", interactive=False)

                # ---------- Tab 2: 实时监控 ----------
                with gr.TabItem("📡 实时监控"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.Markdown("### 🎛️ 仪表盘")
                            # 模拟表盘显示（用进度条）
                            gr.Markdown("**当前负荷**")
                            gr.Slider(value=65, minimum=0, maximum=100, label="容量占比 (%)", interactive=False)
                            gr.Markdown("**绿黄红状态: 🟡 警戒区 (65%)**")
                            gr.Markdown("---")
                            gr.Markdown("**今日预测峰值**")
                            gr.Textbox(value="345 kW @ 14:30", label="", interactive=False)
                            gr.Markdown("**实时净负荷**")
                            gr.Textbox(value="70 kW (P_load - P_pv)", label="", interactive=False)
                        with gr.Column(scale=1):
                            gr.Markdown("### 🌊 功率流向图 (桑基图)")
                            gr.Markdown("*动态显示电力流向（占位）*")
                            gr.Plot(value=None, label="桑基图占位")
                    with gr.Row():
                        gr.Markdown("### 🏷️ 关键指标卡片")
                        gr.Markdown("*绿电替代率: 32.5% &nbsp;&nbsp;|&nbsp;&nbsp; 当前电价: 峰时段 (1.28元/kWh) &nbsp;&nbsp;|&nbsp;&nbsp; 预测偏差 MAPE: 4.2%*")

            
                # ---------- Tab 3: 气象与环境 ----------
                with gr.TabItem("🌤️ 气象与环境"):
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### ☀️ 辐照度与云量")
                            # 替换为动态更新的 Plot
                            radiation_plot = gr.Plot(label="辐照度与云量曲线")
                            
                        with gr.Column():
                            gr.Markdown("### ⚠️ 天气预警")
                            # 替换为动态更新的 HTML
                            warning_html = gr.HTML(value="加载中...")
                            
                            gr.Markdown("---")
                            gr.Markdown("### 📅 未来2小时预报")
                            # 替换为动态更新的 DataFrame
                            forecast_table = gr.Dataframe(
                                headers=["时间", "天气", "辐照度 (W/m²)", "降雨量 (mm)"],
                                interactive=False
                            )
                            
                            # 可选：显示实时数据的小卡片
                            gr.Markdown("---")
                            realtime_info = gr.Markdown("### 实时数据加载中...")
                
                # 添加刷新按钮到气象模块内部（或复用左侧的全局刷新按钮）
                with gr.Row():
                    refresh_weather_btn = gr.Button("🔄 刷新气象数据", variant="secondary")
                
                # 绑定更新事件
                refresh_weather_btn.click(
                    fn=update_weather_ui,
                    inputs=[],
                    outputs=[radiation_plot, warning_html, forecast_table, realtime_info]
                )
                
                # 页面加载时自动刷新一次
                demo.load(fn=update_weather_ui, outputs=[radiation_plot, warning_html, forecast_table, realtime_info])

                # ---------- Tab 4: 策略模拟 ----------
                with gr.TabItem("💡 策略模拟"):
                    gr.Markdown("### 🔋 储能充放电建议")
                    gr.Markdown("*基于净负荷预测与电价时段自动生成*")
                    strategy_data = pd.DataFrame([
                        ["11:00-13:00", "储能充电", "50", "光伏盈余时段，消纳绿电"],
                        ["13:00-15:00", "储能保持", "0", "电价平段，观察负荷"],
                        ["17:00-19:00", "储能放电", "80", "晚高峰+光伏为0，支撑充电桩"]
                    ], columns=["时段", "建议动作", "建议功率 (kW)", "说明"])
                    gr.Dataframe(value=strategy_data, label="今日调度策略", interactive=False)
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### 🔌 充电桩限流建议")
                            gr.Markdown("*预测电力过载时触发*")
                            gr.Textbox(value="当前无过载风险，建议保持当前功率上限", label="限流指令", interactive=False)
                        with gr.Column():
                            gr.Markdown("### 💰 收益分析")
                            gr.Markdown("*光伏直供节省电费*")
                            gr.Textbox(value="今日已节省: 124.50 元", label="节省金额", interactive=False)
                            gr.Markdown("对比电网购电: 节省 23.8%")

    # 底部状态栏
    gr.Markdown("---")
    gr.Markdown("🟢 系统状态: 正常运行 | 数据源: 模拟实时输入 (15min刷新) | 模型服务: 已连接")

# 启动
if __name__ == "__main__":

    demo.launch( 
        share=False
    )

C:\Users\杉嶋桐惠\AppData\Local\Temp\ipykernel_3148\2411608399.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="光储充一体化智能预测平台", css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
